# L02 — The Good and Evil of KV Cache

**Vizuara Inference Engineering Workshop**  
**Instructor:** Dr. Raj  
**Hardware:** NVIDIA H100 80GB HBM3  

This notebook demonstrates:
1. **The autoregressive loop** — naive token-by-token generation (no caching)
2. **KV Cache speedup** — per-token latency benchmark on a 7B model
3. **The dark side** — KV cache memory analysis and the formula
4. **Production inference** — vLLM throughput with batching

---
## 0. Setup

In [ ]:
!pip install -q transformers accelerate matplotlib numpy

In [ ]:
import torch
import torch.nn as nn
import time
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu} ({mem:.0f} GB)")

---
## 1. The Autoregressive Loop — Visualizing the Waste

At every step, the model receives the **entire sequence** and processes all tokens —
but only uses the **last row** of the output to predict the next token.

Watch the input grow with each generated token:

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# We use Qwen2.5-7B — large enough to show real effects on H100
MODEL_NAME = "Qwen/Qwen2.5-7B"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16).cuda().eval()
print("Model loaded!")

In [ ]:
prompt = "The next day is bright"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
input_ids = inputs.input_ids

print(f"Prompt: '{prompt}'")
print(f"Initial tokens: {input_ids.shape[1]}\n")
print("Generated: ", end="")

for step in range(30):
    with torch.no_grad():
        outputs = model(input_ids)              # ← Full forward pass on ALL tokens
    logits = outputs.logits
    next_token_logits = logits[:, -1, :]        # ← Only use the LAST row
    next_token_id = next_token_logits.argmax(dim=-1).unsqueeze(-1)
    input_ids = torch.cat([input_ids, next_token_id], dim=-1)  # Append
    print(tokenizer.decode(next_token_id[0]), end="", flush=True)

print(f"\n\nFinal sequence: {input_ids.shape[1]} tokens")
print("\n⚠️  model(input_ids) is called INSIDE the loop.")
print("   At step 30, we pass 35 tokens through the ENTIRE 7B-param model")
print("   just to get logits for the LAST token. Everything else is wasted!")

---
## 2. KV Cache Speedup — The Real Benchmark

We measure **per-token latency** as the sequence grows. This is the clearest way to see the effect:

- **Without cache:** latency should **increase** with each token (reprocessing everything)
- **With cache:** latency should stay **flat** (only processing the new token)

We use CUDA events for precise GPU-level timing (no Python overhead).

In [ ]:
# Create a long prompt to start from a realistic context length
long_text = ("The history of artificial intelligence began in antiquity with myths "
             "and stories of artificial beings endowed with intelligence. ") * 300

N_DECODE_STEPS = 80  # tokens to generate

# We'll benchmark at different starting context lengths
context_lengths = [512, 1024, 2048, 4096]
all_results = {}

for ctx_len in context_lengths:
    inputs = tokenizer(long_text, return_tensors="pt",
                       max_length=ctx_len, truncation=True).to("cuda")
    base_ids = inputs.input_ids
    actual_len = base_ids.shape[1]

    # === WITHOUT KV Cache ===
    ids = base_ids.clone()
    # Warm up
    with torch.no_grad():
        _ = model(ids[:, :100], use_cache=False)
    torch.cuda.synchronize()

    no_cache_times = []
    for step in range(N_DECODE_STEPS):
        se = torch.cuda.Event(enable_timing=True)
        ee = torch.cuda.Event(enable_timing=True)
        se.record()
        with torch.no_grad():
            out = model(ids, use_cache=False)
        ee.record()
        torch.cuda.synchronize()
        no_cache_times.append(se.elapsed_time(ee))
        next_id = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        ids = torch.cat([ids, next_id], dim=-1)

    # === WITH KV Cache ===
    ids = base_ids.clone()
    # Prefill
    se = torch.cuda.Event(enable_timing=True)
    ee = torch.cuda.Event(enable_timing=True)
    se.record()
    with torch.no_grad():
        out = model(ids, use_cache=True)
    ee.record()
    torch.cuda.synchronize()
    prefill_ms = se.elapsed_time(ee)

    past = out.past_key_values
    next_id = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
    ids = torch.cat([ids, next_id], dim=-1)

    cache_times = []
    for step in range(N_DECODE_STEPS):
        se = torch.cuda.Event(enable_timing=True)
        ee = torch.cuda.Event(enable_timing=True)
        se.record()
        with torch.no_grad():
            out = model(ids[:, -1:], use_cache=True, past_key_values=past)
        ee.record()
        torch.cuda.synchronize()
        cache_times.append(se.elapsed_time(ee))
        past = out.past_key_values
        next_id = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        ids = torch.cat([ids, next_id], dim=-1)

    avg_nc = sum(no_cache_times) / len(no_cache_times)
    avg_c = sum(cache_times) / len(cache_times)
    speedup = avg_nc / avg_c

    all_results[ctx_len] = {
        "no_cache_times": no_cache_times,
        "cache_times": cache_times,
        "prefill_ms": prefill_ms,
        "avg_nc": avg_nc,
        "avg_c": avg_c,
        "speedup": speedup,
    }

    print(f"Context {actual_len:>5d}: "
          f"No Cache {avg_nc:>6.1f} ms/tok | "
          f"Cache {avg_c:>5.1f} ms/tok | "
          f"Prefill {prefill_ms:>5.1f} ms | "
          f"Speedup {speedup:.1f}×")

In [ ]:
# ========== PLOT 1: Per-token latency over time (2048 context) ==========
ctx_key = 2048
r = all_results[ctx_key]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Left: Per-token latency trace
steps = range(len(r["no_cache_times"]))
ax1.plot(steps, r["no_cache_times"], color='#e74c3c', linewidth=1.5,
         alpha=0.8, label=f'Without KV Cache (avg {r["avg_nc"]:.1f} ms)')
ax1.plot(steps, r["cache_times"], color='#2ecc71', linewidth=1.5,
         alpha=0.8, label=f'With KV Cache (avg {r["avg_c"]:.1f} ms)')
ax1.set_xlabel('Decode Step', fontsize=12)
ax1.set_ylabel('Per-Token Latency (ms)', fontsize=12)
ax1.set_title(f'Per-Token Latency — Qwen2.5-7B, {ctx_key} Context\n'
              f'Without cache: GROWING | With cache: FLAT', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Right: Speedup vs context length
ctx_lens = sorted(all_results.keys())
speedups = [all_results[c]["speedup"] for c in ctx_lens]
avg_ncs = [all_results[c]["avg_nc"] for c in ctx_lens]
avg_cs = [all_results[c]["avg_c"] for c in ctx_lens]

x = np.arange(len(ctx_lens))
width = 0.35
bars1 = ax2.bar(x - width/2, avg_ncs, width, label='Without Cache',
                color='#e74c3c', alpha=0.8)
bars2 = ax2.bar(x + width/2, avg_cs, width, label='With Cache',
                color='#2ecc71', alpha=0.8)

# Add speedup labels
for i, (nc, c, s) in enumerate(zip(avg_ncs, avg_cs, speedups)):
    ax2.annotate(f'{s:.1f}×', xy=(i, max(nc, c) + 3),
                ha='center', fontsize=13, fontweight='bold', color='#2c3e50')

ax2.set_xticks(x)
ax2.set_xticklabels([str(c) for c in ctx_lens])
ax2.set_xlabel('Starting Context Length (tokens)', fontsize=12)
ax2.set_ylabel('Avg Per-Token Latency (ms)', fontsize=12)
ax2.set_title('KV Cache Speedup Grows with Context Length\n'
              'Qwen2.5-7B on H100', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('kv_cache_speedup.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Key insights:")
print("   1. Without cache: latency GROWS because we reprocess everything")
print("   2. With cache: latency stays FLAT — only processes the new token")
print("   3. Speedup scales linearly with context length")
print(f"   4. At 4096 context: {all_results[4096]['speedup']:.1f}× speedup!")

---
## 3. The Dark Side — KV Cache Memory Analysis

The KV cache trades **compute** for **memory**. Let's quantify exactly how much memory it consumes.

### The Formula

```
KV Cache Size (bytes) = 2 × layers × batch_size × kv_heads × head_dim × seq_len × bytes_per_param
                        │                         │                               │
                        K and V                   GQA reduces this                2 for fp16
```

In [ ]:
def kv_cache_size_gb(layers, kv_heads, head_dim, seq_len,
                     batch_size=1, bytes_per_param=2):
    """Calculate KV cache size in GB."""
    # 2 for K and V matrices
    size_bytes = 2 * layers * batch_size * kv_heads * head_dim * seq_len * bytes_per_param
    return size_bytes / (1024**3)


# Real model specifications
models = {
    "GPT-2 (124M)":          dict(layers=12,  kv_heads=12,  head_dim=64,  seq_len=1024),
    "Llama 3 8B (GQA)":      dict(layers=32,  kv_heads=8,   head_dim=128, seq_len=8192),
    "Qwen2.5-7B (GQA)":      dict(layers=28,  kv_heads=4,   head_dim=128, seq_len=8192),
    "GPT-3 (175B, MHA)":     dict(layers=96,  kv_heads=96,  head_dim=128, seq_len=4096),
    "Llama 3 70B (GQA)":     dict(layers=80,  kv_heads=8,   head_dim=128, seq_len=8192),
    "DeepSeek-V3 (671B)":    dict(layers=61,  kv_heads=128, head_dim=128, seq_len=128000),
}

print(f"{'Model':<25} {'b=1':>10} {'b=32':>10} {'b=128':>12}")
print("─" * 60)
for name, cfg in models.items():
    s1 = kv_cache_size_gb(**cfg, batch_size=1)
    s32 = kv_cache_size_gb(**cfg, batch_size=32)
    s128 = kv_cache_size_gb(**cfg, batch_size=128)
    print(f"{name:<25} {s1:>8.2f} GB {s32:>8.1f} GB {s128:>9.1f} GB")

print("\n⚠️  DeepSeek-V3 at 128K context: 478 GB for a SINGLE sequence!")
print("   This is why DeepSeek invented Multi-Head Latent Attention (MLA).")

### 3.1 — Measuring the KV Cache Live on GPU

Let's inspect the actual KV cache tensors that HuggingFace stores during inference.

In [ ]:
# Inspect the KV cache object from our model
long_text = ("The history of artificial intelligence began in antiquity "
             "with myths and stories of artificial beings endowed with intelligence. ") * 500

print("=== KV Cache Structure (Qwen2.5-7B) ===\n")

for ctx_len in [1024, 2048, 4096, 8192]:
    inputs = tokenizer(long_text, return_tensors="pt",
                       max_length=ctx_len, truncation=True).to("cuda")
    torch.cuda.empty_cache()

    with torch.no_grad():
        out = model(inputs.input_ids, use_cache=True)
    past = out.past_key_values

    # Measure total KV cache memory (DynamicCache yields tuples per layer)
    kv_bytes = 0
    n_layers = 0
    for layer_tuple in past:
        n_layers += 1
        for tensor in layer_tuple:
            if tensor is not None and hasattr(tensor, 'nelement'):
                kv_bytes += tensor.nelement() * tensor.element_size()

    actual_seq = inputs.input_ids.shape[1]
    # Formula prediction (Qwen2.5-7B: 28 layers, 4 KV heads, head_dim 128)
    predicted = kv_cache_size_gb(layers=28, kv_heads=4, head_dim=128,
                                 seq_len=actual_seq) * 1024  # in MB

    print(f"Context {actual_seq:>5d}: Measured {kv_bytes/1e6:>6.1f} MB | "
          f"Formula {predicted:>6.1f} MB | "
          f"Layers: {n_layers}")

    if ctx_len == 1024:
        # Show shape of first layer's K tensor
        first_layer = list(past)[0]
        print(f"  K shape: {first_layer[0].shape}  →  (batch, kv_heads, seq_len, head_dim)")

    del out, past
    torch.cuda.empty_cache()

print(f"\n✅ Formula matches live measurement!")
print(f"   KV cache grows linearly with sequence length — no escape.")

In [ ]:
# ========== PLOT 2: KV cache size vs context length ==========
seq_lengths = np.arange(512, 65537, 512)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Left: KV cache growth per model
plot_models = {
    "Qwen2.5-7B (GQA, 4 KV heads)": dict(layers=28, kv_heads=4, head_dim=128),
    "Llama 3 8B (GQA, 8 KV heads)":  dict(layers=32, kv_heads=8, head_dim=128),
    "Llama 3 70B (GQA, 8 KV heads)": dict(layers=80, kv_heads=8, head_dim=128),
    "GPT-3 175B (MHA, 96 heads)":     dict(layers=96, kv_heads=96, head_dim=128),
}
colors = ['#2ecc71', '#3498db', '#e67e22', '#e74c3c']

for (name, cfg), color in zip(plot_models.items(), colors):
    sizes = [kv_cache_size_gb(**cfg, seq_len=s) for s in seq_lengths]
    ax1.plot(seq_lengths / 1000, sizes, linewidth=2, label=name, color=color)

ax1.axhline(y=80, color='gray', linestyle='--', linewidth=1.5, alpha=0.7)
ax1.text(62, 83, 'H100 80GB', fontsize=10, color='gray')
ax1.set_xlabel('Sequence Length (K tokens)', fontsize=12)
ax1.set_ylabel('KV Cache Size (GB) — batch=1', fontsize=12)
ax1.set_title('The Dark Side: KV Cache Memory Growth', fontsize=13)
ax1.legend(fontsize=10, loc='upper left')
ax1.grid(True, alpha=0.3)

# Right: Max concurrent users on H100
model_mem_gb = 14  # Qwen2.5-7B fp16
overhead_gb = 2
available = 80 - model_mem_gb - overhead_gb

ctx_options = [1024, 2048, 4096, 8192, 16384, 32768]
users_qwen = []
users_llama8b = []
for ctx in ctx_options:
    kv_qwen = kv_cache_size_gb(layers=28, kv_heads=4, head_dim=128, seq_len=ctx)
    kv_llama = kv_cache_size_gb(layers=32, kv_heads=8, head_dim=128, seq_len=ctx)
    users_qwen.append(int(available / kv_qwen) if kv_qwen > 0 else 0)
    users_llama8b.append(int(available / kv_llama) if kv_llama > 0 else 0)

x = np.arange(len(ctx_options))
width = 0.35
ax2.bar(x - width/2, users_qwen, width, label='Qwen2.5-7B', color='#2ecc71', alpha=0.8)
ax2.bar(x + width/2, users_llama8b, width, label='Llama 3 8B', color='#3498db', alpha=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels([f'{c//1024}K' for c in ctx_options])
ax2.set_xlabel('Context Length', fontsize=12)
ax2.set_ylabel('Max Concurrent Users', fontsize=12)
ax2.set_title('Max Users on H100 (80GB)\nDoubling context = halving users', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, axis='y')
for i, (q, l) in enumerate(zip(users_qwen, users_llama8b)):
    ax2.text(i - width/2, q + 5, str(q), ha='center', fontsize=9)
    ax2.text(i + width/2, l + 5, str(l), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('kv_cache_dark_side.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💡 Doubling context length HALVES your max concurrent users.")
print("   This is why API providers charge more for larger context windows.")
print("   The hardware cost is dominated by KV cache memory, not compute.")

### 3.2 — The Memory Budget on H100

On a real GPU, memory is shared between:
- **Model weights** (fixed)
- **KV cache** (grows with context × batch)
- **Activations** (temporary)

The KV cache determines how many users you can serve simultaneously.

In [ ]:
def analyze_serving(gpu_mem_gb, model_name, model_mem_gb, layers, kv_heads,
                    head_dim, overhead_gb=2.0):
    """Analyze serving capacity for a model on a GPU."""
    available = gpu_mem_gb - model_mem_gb - overhead_gb
    print(f"\n{'='*60}")
    print(f"{model_name} on {gpu_mem_gb}GB GPU")
    print(f"  Model weights: {model_mem_gb} GB")
    print(f"  Overhead:      {overhead_gb} GB")
    print(f"  Available:     {available:.0f} GB for KV cache")
    print(f"{'─'*60}")
    print(f"  {'Context':<10} {'KV/user':>10} {'Max Users':>12} {'Total KV':>10}")
    print(f"{'─'*60}")
    for ctx in [2048, 4096, 8192, 32768, 131072]:
        kv_per_user = kv_cache_size_gb(layers, kv_heads, head_dim, ctx)
        max_users = int(available / kv_per_user) if kv_per_user > 0 else 0
        total_kv = kv_per_user * max_users
        print(f"  {ctx:<10d} {kv_per_user*1000:>8.0f} MB {max_users:>10d}   {total_kv:>8.1f} GB")


analyze_serving(80, "Qwen2.5-7B (fp16)", 14,
                layers=28, kv_heads=4, head_dim=128)

analyze_serving(80, "Llama 3 8B (fp16)", 16,
                layers=32, kv_heads=8, head_dim=128)

analyze_serving(80, "Llama 3 70B (fp16, TP=4, per-GPU)", 35,
                layers=80, kv_heads=8, head_dim=128)

print("\n\n🔑 KEY TAKEAWAY:")
print("   The KV cache — not model weights — is the #1 bottleneck for serving.")
print("   This is why the ENTIRE next lecture (L03) is about reducing it.")

---
## 4. Production Inference with vLLM

vLLM is the production inference engine we'll use throughout this course.
It uses:
- **KV Cache** (always on — you can't turn it off!)
- **PagedAttention** — manages KV cache like OS virtual memory (no fragmentation)
- **Continuous Batching** — dynamically batches requests for throughput

Let's see what production-grade inference looks like.

In [ ]:
!pip install -q vllm 2>&1 | tail -3

In [ ]:
# Free GPU memory from HuggingFace model
del model
torch.cuda.empty_cache()
import gc; gc.collect()
print("Freed GPU memory for vLLM.")

In [ ]:
from vllm import LLM, SamplingParams

# Load the same model in vLLM
llm = LLM(model="Qwen/Qwen2.5-7B", dtype="float16", max_model_len=8192)
sampling_params = SamplingParams(max_tokens=200, temperature=0)

prompt = ("The history of artificial intelligence began in antiquity with "
          "myths and stories of artificial beings. The seeds of modern AI "
          "were planted by classical philosophers")

print("=== vLLM Throughput Benchmark ===\n")
print(f"Model: Qwen2.5-7B | GPU: H100 | Max tokens: 200\n")

results_vllm = []
for batch_size in [1, 8, 32, 64]:
    prompts = [prompt] * batch_size
    start = time.time()
    outputs = llm.generate(prompts, sampling_params)
    elapsed = time.time() - start

    total_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
    throughput = total_tokens / elapsed
    per_request_ms = (elapsed / batch_size) * 1000

    results_vllm.append((batch_size, elapsed, total_tokens, throughput, per_request_ms))
    print(f"  Batch {batch_size:>3d}: {elapsed:.2f}s | "
          f"{total_tokens:>5d} tokens | "
          f"{throughput:>6.0f} tok/s | "
          f"{per_request_ms:>5.0f} ms/req")

print(f"\n💡 vLLM scales from {results_vllm[0][3]:.0f} to "
      f"{results_vllm[-1][3]:.0f} tok/s — {results_vllm[-1][3]/results_vllm[0][3]:.0f}× "
      f"throughput with batching!")
print("   This is the power of KV cache + PagedAttention + continuous batching.")
print("   We will study PagedAttention in L05 and continuous batching in L07.")

In [ ]:
# Plot vLLM throughput scaling
batch_sizes = [r[0] for r in results_vllm]
throughputs = [r[3] for r in results_vllm]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(range(len(batch_sizes)), throughputs, color='#9b59b6', alpha=0.85)
ax.set_xticks(range(len(batch_sizes)))
ax.set_xticklabels([str(b) for b in batch_sizes])
ax.set_xlabel('Batch Size (concurrent requests)', fontsize=12)
ax.set_ylabel('Throughput (tokens/sec)', fontsize=12)
ax.set_title('vLLM Throughput Scaling — Qwen2.5-7B on H100\n'
             'KV Cache + PagedAttention + Continuous Batching', fontsize=13)
for bar, t in zip(bars, throughputs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{t:.0f}', ha='center', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('vllm_throughput.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Summary

### The Good (Compute Savings)

| Metric | Without KV Cache | With KV Cache |
|--------|:---:|:---:|
| Attention FLOPs (total) | O(T³) | O(T²) |
| Per-token latency | **Grows** with position | **Flat** |
| Speedup at 4K context | — | **~9×** |
| Output quality | Same | Same |

### The Evil (Memory Cost)

```
KV Cache = 2 × layers × batch × kv_heads × head_dim × seq_len × 2 bytes
```

| Model | KV Cache (single sequence, max context) |
|-------|:---:|
| GPT-2 (124M) | 36 MB |
| Llama 3 8B | 1 GB |
| GPT-3 (175B) | 18 GB |
| DeepSeek-V3 | 478 GB |

### What's Next (L03)

How do we tame the KV cache?
- **MQA** — share K,V across all heads → cache ÷ n
- **GQA** — share K,V within groups → tunable compromise
- **MLA** — compress K,V into latent space → best of both worlds